In [2]:
import cv2
import torch
import torchvision.transforms as transforms
from torchvision import datasets
import torch.nn as nn
import os
from PIL import Image
import numpy as np

# === Configuration ===
MODEL_PATH = "best_asl_model.pth"
LABELS_PATH = "asl_labels.txt"  # Must contain class names in order (e.g., ['A', 'B', ...])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 64

# === Load class labels ===
# with open(LABELS_PATH, "r") as f:
#     classes = [line.strip() for line in f.readlines()]
dataset = datasets.ImageFolder("C:/Users/kusal/OneDrive/Documents/Uni stuff/Year 5 Sem 1/FYP-A/asl-alphabet/train")
classes = dataset.classes

# === Define Model (must match training) ===
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=26):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCNN(num_classes=len(classes)).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

# === Define Image Transform ===
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

# === Start Webcam ===
cap = cv2.VideoCapture(0)
print("Press 'q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Center crop square
    h, w, _ = frame.shape
    min_dim = min(h, w)
    start_x = w//2 - min_dim//2
    start_y = h//2 - min_dim//2
    cropped = frame[start_y:start_y+min_dim, start_x:start_x+min_dim]

    # Convert for model
    image = Image.fromarray(cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB))
    input_tensor = transform(image).unsqueeze(0).to(DEVICE)

    # Predict
    with torch.no_grad():
        output = model(input_tensor)
        pred_idx = torch.argmax(output, dim=1).item()
        label = classes[pred_idx]

    # Display
    cv2.putText(frame, f"Prediction: {label}", (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,255,0), 3)
    cv2.imshow("ASL Sign Recognition", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Press 'q' to quit.
